Đây là script chạy huấn luyện mô hình đề xuất gồm các bước:
- Clone repository từ GitHub
- Load dữ liệu đề bài của cuộc thi
- Huấn luyện mô hình
- Lưu kết quả huấn luyện và file submission

**Lưu ý**:
- Script này **không thể** chạy local do ngốn bộ nhớ. Script **bắt buộc** phải chạy trên GPU, chúng tôi có đính kèm hướng dẫn chi tiết chạy trên [Kaggle](./HOW_TO_RUN_KAGGLE.md).
- Notebook chạy cho hai việc:
    - Huấn luyện các mô hình đề xuất (2 mô hình)
    - Chạy nghiên cứu cắt bỏ (3 trường hợp)
- Thời gian chạy và kết quả cụ thể cho từng mô hình có thể xem tại [đây](../model/MODEL_RESULTS.md)
- Với những trường hợp có thời gian chạy quá lớn (>30 phút), chúng tôi khuyến khích nên sử dụng cơ chế chạy offline trên Kaggle.
- Cell code 9 là cell cần phải chú ý, trước khi chạy bạn phải:
    - Thay đổi đường dẫn đến file data (Do nó thay đổi tùy theo tài khoản Kaggle)
    - Chọn loại mô hình muốn chạy
    - Chọn loại nghiên cứu cắt bỏ (mặc định sử dụng mô hình TAGNet nếu chạy nghiên cứu cắt bỏ):
        - "none": Huấn luyện đầy đủ mô hình
        - "stage 1": Cắt bỏ giai đoạn 1 quá trình huấn luyện
        - "stage 2": Cắt bỏ giai đoạn 2 quá trình huấn luyện
        - "max penalty loss": Cắt bỏ hàm loss tùy chỉnh

In [ ]:
!git clone https://github.com/CryAndRRich/tagflow.git

In [ ]:
%cd /kaggle/working/tagflow
TAGFLOW_PATH = "/kaggle/working/tagflow"

In [ ]:
!pip install -r requirements.txt

In [3]:
import sys

if TAGFLOW_PATH not in sys.path:
    sys.path.append(TAGFLOW_PATH)

In [4]:
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
from torch.amp import autocast

In [5]:
from config import *
from preprocess import *
from model import *
from utils import *

In [ ]:
# Hàm set_seed định nghĩa tại utils/set_up.py
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [7]:
INPUT_ROOT = "YOUR INPUT ROOT PATH"
WORK_DIR = "/kaggle/working"
SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"

# MODEL_NAME có thể nhận các giá trị là: tagnet, tacnet, tarnet, taanet
MODEL_NAME = "taenet"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

# ABLATION_STUDY có thể nhận các giá trị là: none, stage 1, stage 2, max penalty loss
ABLATION_STUDY = "none"

In [8]:
# Class DataManager được định nghĩa tại preprocess/__init__.py
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [9]:
if MODEL_NAME == "taanet":
    PRETRAIN_FILE = f"{WORK_DIR}/{MODEL_NAME}_pretrain_model.pth"
    masked_loader = data_manager.get_dataloaders(masked=True)
    # Hàm get_model được định nghĩa tại model/models/__init__.py
    pretrain_model = get_model(name="pretrain_taanet", vocab_size=data_manager.VOCAB_SIZE)
    # Hàm train_pretrain_model được định nghĩa tại model/train.py
    train_pretrain_model(
        model=pretrain_model,
        dataloader=masked_loader,
        pretrain_file=PRETRAIN_FILE,
        device=DEVICE,
        verbose=True
    )

In [10]:
x_train, y_train, x_val, y_val, x_test = data_manager.get_data()
train_loader_stage_1, val_loader_stage_1, test_loader_stage_1 = data_manager.get_dataloaders()

In [ ]:
if ABLATION_STUDY != "stage 1":
    y_val_pseudo = pd.DataFrame({"id": x_val["id"]})
    y_test_pseudo = pd.DataFrame({"id": x_test["id"]})
    
    # Hàm get_loss_functions được định nghĩa tại utils/prepare_model.py
    loss_fns_stage_1 = get_loss_functions(
        y_df=y_train,
        attribute_cols=data_manager.ATTRIBUTE_COLS,
        num_classes_list=data_manager.NUM_CLASSES_LIST,
        label_smoothing=CONFIG_MODEL.LABEL_SMOOTHING,
        device=DEVICE
    )
    
    for attr_idx, attr_col in enumerate(data_manager.ATTRIBUTE_COLS):
        scheduler_kwargs_stage_1 = deepcopy(CONFIG_MODEL.SCHEDULER_KWARGS)
        scheduler_kwargs_stage_1["epochs"] = CONFIG_MODEL.NUM_EPOCHS_STAGE_1
        scheduler_kwargs_stage_1["steps_per_epoch"] = len(train_loader_stage_1)
        
        # Hàm get_model_optim_schedule được định nghĩa tại utils/prepare_model.py
        model_stage_1, optimizer_stage_1, scheduler_stage_1 = get_model_optim_schedule(
            model_name=MODEL_NAME,
            data=data_manager,
            attribute_idx=attr_idx,
            model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME],
            optim_kwargs=CONFIG_MODEL.OTIMIZER_KWARGS,
            scheduler_kwargs=scheduler_kwargs_stage_1,
            device=DEVICE
        )
    
        # Hàm train_model_stage_1 được định nghĩa tại model/train.py
        model_stage_1 = train_model_stage_1(
            model=model_stage_1, 
            train_loader=train_loader_stage_1, 
            val_loader=val_loader_stage_1, 
            loss_fn=loss_fns_stage_1[attr_idx], 
            optimizer=optimizer_stage_1, 
            scheduler=scheduler_stage_1,
            attribute_idx=attr_idx, 
            num_epochs=CONFIG_MODEL.NUM_EPOCHS_STAGE_1,
            early_stopping=CONFIG_MODEL.EARLY_STOPPING_STAGE_1,
            pretrain_file=PRETRAIN_FILE,
            device=DEVICE,
            verbose=True
        )

        if ABLATION_STUDY == "stage 2":
            model_stage_1.eval()
            preds_stage_1 = []
            with torch.no_grad():
                for batch_x, _ in val_loader_stage_1:
                    batch_x = batch_x.to(DEVICE)
                    with autocast("cuda"):
                        out = model_stage_1(batch_x)
                    preds_stage_1.extend(torch.argmax(out[0], dim=1).cpu().numpy())
                    
            y_val_pseudo[attr_col] = preds_stage_1
        
        model_stage_1.eval()
        preds_stage_1 = []
        with torch.no_grad():
            for batch_x in test_loader_stage_1:
                batch_x = batch_x.to(DEVICE)
                with autocast("cuda"):
                    out = model_stage_1(batch_x)
                preds_stage_1.extend(torch.argmax(out[0], dim=1).cpu().numpy())
                
        y_test_pseudo[attr_col] = preds_stage_1

    if ABLATION_STUDY == "none":
        data_manager.add_data(
            extra_x=x_test, 
            extra_y=y_test_pseudo
        )

In [12]:
train_loader_stage_2, val_loader_stage_2, test_loader_stage_2 = data_manager.get_dataloaders()

In [ ]:
if ABLATION_STUDY != "stage 2":
    loss_fns_stage_2 = get_loss_functions(
        y_df=y_train,
        attribute_cols=data_manager.ATTRIBUTE_COLS,
        num_classes_list=data_manager.NUM_CLASSES_LIST,
        label_smoothing=CONFIG_MODEL.LABEL_SMOOTHING,
        device=DEVICE
    )
    
    scheduler_kwargs_stage_2 = deepcopy(CONFIG_MODEL.SCHEDULER_KWARGS)
    scheduler_kwargs_stage_2["epochs"] = CONFIG_MODEL.NUM_EPOCHS_STAGE_2
    scheduler_kwargs_stage_2["steps_per_epoch"] = len(train_loader_stage_2)
    
    model_stage_2, optimizer_stage_2, scheduler_stage_2 = get_model_optim_schedule(
        model_name=MODEL_NAME,
        data=data_manager,
        attribute_idx=None,
        model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME],
        optim_kwargs=CONFIG_MODEL.OTIMIZER_KWARGS,
        scheduler_kwargs=scheduler_kwargs_stage_2,
        device=DEVICE
    )

    if ABLATION_STUDY == "max penalty loss":
        alpha_max_loss = 0.0
    else:
        alpha_max_loss = CONFIG_MODEL.ALPHA_MAX_LOSS
    
    # Hàm train_model_stage_2 được định nghĩa tại model/train.py
    final_model = train_model_stage_2(
        model=model_stage_2, 
        train_loader=train_loader_stage_2, 
        val_loader=val_loader_stage_2, 
        loss_fns=loss_fns_stage_2, 
        optimizer=optimizer_stage_2, 
        scheduler=scheduler_stage_2,
        attribute_list=data_manager.ATTRIBUTE_LIST, 
        num_epochs=CONFIG_MODEL.NUM_EPOCHS_STAGE_2,
        alpha_max_loss=alpha_max_loss,
        early_stopping=CONFIG_MODEL.EARLY_STOPPING_STAGE_2,
        checkpoint_file=CHECKPOINT_FILE,
        pretrain_file=PRETRAIN_FILE,
        device=DEVICE,
        verbose=True
    )

In [14]:
if ABLATION_STUDY != "stage 2":
    _, val_loader_final, test_loader_final = data_manager.get_dataloaders()

    # Hàm run_inference được định nghĩa tại utils/evaluate.py
    val_predictions = run_inference(
        model=final_model, 
        loader=val_loader_final,
        attribute_list=data_manager.ATTRIBUTE_LIST,
        device=DEVICE
    )
    
    test_predictions = run_inference(
        model=final_model, 
        loader=test_loader_final,
        attribute_list=data_manager.ATTRIBUTE_LIST,
        device=DEVICE
    )

else:
    val_predictions = {}
    test_predictions = {}
    
    for col, attr_id in zip(data_manager.ATTRIBUTE_COLS, data_manager.ATTRIBUTE_LIST):
        dict_key = f"attr_{attr_id}"
        val_predictions[dict_key] = y_val_pseudo[col].tolist()
        test_predictions[dict_key] = y_test_pseudo[col].tolist()

In [ ]:
get_stats(
    val_predictions=val_predictions,
    y_true=y_val,
    attribute_list=data_manager.ATTRIBUTE_LIST,
    attribute_cols=data_manager.ATTRIBUTE_COLS
)

In [ ]:
submission_df = pd.DataFrame()
submission_df["id"] = x_test["id"]

for col in data_manager.ATTRIBUTE_COLS:
    submission_df[col] = np.array(test_predictions[col], dtype=np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)

print(f"File nộp bài được lưu tại: {SUBMISSION_FILE}")
print(submission_df)
print(submission_df.dtypes)